In [0]:
# Define notebook parameters (widgets); values are passed in by the calling job/notebook
# landing_timestamp : bronze batch to process, e.g. 2026-11-05_10:37:19
# table_name        : source/target table name, e.g. hospitals
# merge_key         : key column(s) for MERGE, comma-separated for composite keys, e.g. hospital_id
# load_type         : FULL (overwrite), APPEND or MERGE (upsert); defaults to FULL when empty
dbutils.widgets.text("landing_timestamp", "")
dbutils.widgets.text("table_name", "")
dbutils.widgets.text("merge_key", "")
dbutils.widgets.text("load_type", "")

# Read parameter values
landing_timestamp = dbutils.widgets.get("landing_timestamp")
table_name = dbutils.widgets.get("table_name")
merge_key = dbutils.widgets.get("merge_key")
load_type = dbutils.widgets.get("load_type")

# Log the received parameters
print("=" * 80)
print("[STEP 1] Notebook parameters received")
print(f"  landing_timestamp : '{landing_timestamp}'")
print(f"  table_name        : '{table_name}'")
print(f"  merge_key         : '{merge_key}'")
print(f"  load_type         : '{load_type}'")
print("=" * 80)

In [0]:
# Read only the bronze batch that landed at the given landing_timestamp
print("[STEP 2] Reading bronze batch")
print(f"  Source table : adb_caresync.bronze.{table_name}")
print(f"  Equivalent query:\n    SELECT * FROM adb_caresync.bronze.{table_name} WHERE landing_timestamp = '{landing_timestamp}'")
bronze_df = spark.read.table(f"adb_caresync.bronze.{table_name}").filter(f"landing_timestamp = '{landing_timestamp}'")

# Log what was read (count triggers a Spark job on the bronze batch)
print(f"  Rows read from bronze : {bronze_df.count()}")
print(f"  Columns ({len(bronze_df.columns)}) : {bronze_df.columns}")
print("=" * 80)

In [0]:
from pyspark.sql import functions as F

# Target silver table for this load
target_table = f"adb_caresync.silver.{table_name}"

# Normalize load_type (case/whitespace insensitive); empty value falls back to FULL
normalized_load_type = (load_type or "FULL").strip().upper()

# Split merge_key into a list (supports composite keys such as "id1,id2")
merge_key_list = [key.strip() for key in merge_key.split(",") if key.strip()]

print("[STEP 3] Preparing load metadata")
print(f"  Target table         : {target_table}")
print(f"  Normalized load_type : {normalized_load_type}")
print(f"  Merge key list       : {merge_key_list}")

# MERGE needs at least one key, and every key must exist in the source data
if normalized_load_type == "MERGE":
    if not merge_key_list:
        raise ValueError("merge_key must be provided when load_type is 'MERGE'.")

    missing_merge_keys = [key for key in merge_key_list if key not in bronze_df.columns]
    if missing_merge_keys:
        raise ValueError(f"merge_key not found in source data: {missing_merge_keys}")
    print(f"  Merge key validation passed: all keys {merge_key_list} exist in source data")

# Add audit columns: insert_timestamp (when the row was first loaded) and update_timestamp (last change)
updated_df = (
    bronze_df.withColumn("insert_timestamp", F.current_timestamp())
    .withColumn("update_timestamp", F.current_timestamp())
)

print("  Added audit columns : insert_timestamp, update_timestamp")
print(f"  Final columns ({len(updated_df.columns)}) : {updated_df.columns}")
print("=" * 80)

In [0]:
# target_table, normalized_load_type and merge_key_list are prepared in the previous cell
print(f"[STEP 4] Writing to silver table {target_table} using load_type = {normalized_load_type}")

if normalized_load_type == "FULL":
    # FULL: replace the whole target table with the current batch
    print(f"  FULL load: overwriting {target_table} with the current batch")
    print(f"  Equivalent operation:\n    CREATE OR REPLACE TABLE {target_table} AS SELECT * FROM <bronze batch + audit columns>")
    updated_df.write \
        .mode("overwrite") \
        .saveAsTable(target_table)
    print(f"  FULL load completed for {target_table}")

elif normalized_load_type == "APPEND":
    # APPEND: add the current batch to the existing rows (creates the table if missing)
    print(f"  APPEND load: appending current batch to {target_table}")
    print(f"  Equivalent operation:\n    INSERT INTO {target_table} SELECT * FROM <bronze batch + audit columns>")
    updated_df.write \
        .mode("append") \
        .saveAsTable(target_table)
    print(f"  APPEND load completed for {target_table}")

elif normalized_load_type == "MERGE":
    if not spark.catalog.tableExists(target_table):
        # First MERGE run: target does not exist yet, so create it from the batch
        print(f"  MERGE load: target {target_table} does not exist yet, creating it from the current batch")
        updated_df.write.mode("overwrite").saveAsTable(target_table)
        print(f"  Initial table creation completed for {target_table}")
    else:
        # Register the batch as a temp view so it can be used in the SQL MERGE
        source_view = "silver_merge_source"
        updated_df.createOrReplaceTempView(source_view)
        print(f"  MERGE load: target {target_table} exists, registered source temp view '{source_view}'")

        # Join condition on the merge key(s), e.g. target.`hospital_id` = source.`hospital_id`
        merge_condition = " AND ".join(
            [f"target.`{key}` = source.`{key}`" for key in merge_key_list]
        )
        # Columns to update on a match; insert_timestamp is kept from the original insert
        update_assignments = ", ".join(
            [
                f"target.`{column}` = source.`{column}`"
                for column in updated_df.columns
                if column != "insert_timestamp"
            ]
        )
        # Column list and values for new rows
        insert_columns = ", ".join([f"`{column}`" for column in updated_df.columns])
        insert_values = ", ".join([f"source.`{column}`" for column in updated_df.columns])

        print(f"  Merge condition    : {merge_condition}")
        print(f"  Update assignments : {update_assignments}")
        print(f"  Insert columns     : {insert_columns}")
        print(f"  Insert values      : {insert_values}")

        # Upsert: update matching rows, insert new rows
        # Build the MERGE statement in a variable so it can be printed before execution
        merge_sql = f"""
            MERGE INTO {target_table} AS target
            USING {source_view} AS source
            ON {merge_condition}
            WHEN MATCHED THEN
              UPDATE SET {update_assignments}
            WHEN NOT MATCHED THEN
              INSERT ({insert_columns})
              VALUES ({insert_values})
        """
        print("  MERGE query prepared:")
        print(merge_sql)

        merge_result_df = spark.sql(merge_sql)
        # MERGE returns a row with affected/updated/deleted/inserted row counts
        print("  MERGE result metrics:")
        merge_result_df.show(truncate=False)
        print(f"  MERGE load completed for {target_table}")

else:
    # Any other load_type value is not supported
    raise ValueError(f"Invalid load_type '{normalized_load_type}'. Expected FULL, APPEND or MERGE")

print("=" * 80)

In [0]:
%sql
-- Preview the silver table
select * from adb_caresync.silver.hospitals

In [0]:
# Return the number of records processed in this batch to the calling job/notebook
record_count = updated_df.count()
print(f"[STEP 5] Records processed in this batch: {record_count}")
print("  Exiting notebook and returning record count to the caller")
print("=" * 80)
dbutils.notebook.exit(str(record_count))